# Phase 2: Data Cleaning, Synchronous Merging, and Hourly Aggregation

**Project Title:** Workload-Aware SARIMA-GARCH Framework for Data Center Cooling Load Forecasting and Uncertainty Analysis  
**Course Module:** Time Series Analysis  
**Syllabus Alignment:** Unit 1 (Data Representation, Regular Intervals, Temporal Smoothing) & Unit 3 (Sampling Interval Uniformity, Missingness Mechanisms)  
**Primary Dataset:** NREL ESIF Data Center Telemetry (May 1, 2024 – April 30, 2025)  

---

### Objectives of Phase 2:
1. Parse raw timestamp strings into structured date-time objects without timezone ambiguity.
2. Detect and purge genuine duplicate timestamps.
3. Treat non-physical telemetry readings (e.g., zero cooling demand during active operation, extreme PUE spikes).
4. Perform minute-aligned synchronous inner joining of PUE and Weather feeds to avoid cartesian/many-to-many join hazards.
5. Aggregate high-frequency 1-minute telemetry into a uniform **Hourly (1H)** time-series index ($\Delta t = 1\text{ hour}$, $N = 8,760$).
6. Handle brief sensor dropouts using temporal linear interpolation without lookahead bias.
7. Export cleaned, merged, and hourly datasets to `../data/processed/`.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ['MPLCONFIGDIR'] = os.path.abspath('../scratch')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

## 1. Loading Raw 1-Year Telemetry Subsets

We ingest the 1-year continuous subsets established in Phase 1:
- `pue_subset_2024may_2025apr.csv`
- `weather_subset_2024may_2025apr.csv`

In [ ]:
raw_dir = os.path.abspath('../data/raw')
pue_path = os.path.join(raw_dir, 'pue_subset_2024may_2025apr.csv')
weather_path = os.path.join(raw_dir, 'weather_subset_2024may_2025apr.csv')

df_pue_raw = pd.read_csv(pue_path)
df_weather_raw = pd.read_csv(weather_path)

print(f"Raw PUE Observations:     {len(df_pue_raw):,}")
print(f"Raw Weather Observations: {len(df_weather_raw):,}")

## 2. Timestamp Parsing & Deduplication

Timestamps in both telemetry streams are formatted as JavaScript date strings containing Mountain Time indicators:
`Day Mon DD YYYY HH:MM:SS GMT-0700 (Mountain Standard Time)`

We parse the fixed 24-character substring (`%a %b %d %Y %H:%M:%S`) to establish local wall-clock timestamps. Duplicate records (timestamps with identical sensor readings or sensor chatter) are identified and purged, retaining the first occurrence.

In [ ]:
df_pue = df_pue_raw.copy()
df_weather = df_weather_raw.copy()

df_pue['datetime'] = pd.to_datetime(df_pue['ts'].str[:24], format='%a %b %d %Y %H:%M:%S', errors='coerce')
df_weather['datetime'] = pd.to_datetime(df_weather['ts'].str[:24], format='%a %b %d %Y %H:%M:%S', errors='coerce')

# Verify zero unparseable timestamps
assert df_pue['datetime'].isna().sum() == 0, "Unparseable timestamps found in PUE!"
assert df_weather['datetime'].isna().sum() == 0, "Unparseable timestamps found in Weather!"

# Deduplicate
pue_dups = df_pue['datetime'].duplicated().sum()
weather_dups = df_weather['datetime'].duplicated().sum()
print(f"PUE duplicate timestamps removed:     {pue_dups:,} ({pue_dups/len(df_pue)*100:.3f}%)")
print(f"Weather duplicate timestamps removed: {weather_dups:,} ({weather_dups/len(df_weather)*100:.3f}%)")

df_pue = df_pue.drop_duplicates(subset=['datetime'], keep='first').sort_values('datetime')
df_weather = df_weather.drop_duplicates(subset=['datetime'], keep='first').sort_values('datetime')

## 3. Physical Telemetry Cleaning & Anomaly Treatment

1. **Discard Uninformative Columns**:
   - `tags`: 100% null metadata column.
   - `unit`: Constant string (`'outside'`).
   - `day`: Redundant scalar.
   - `energy_reuse`: 33.4% negative measurements, deemed physically ungrounded.

2. **Electrical Cooling Demand Zeros (`cooling_kw`)**:
   - Exactly 3 records have `cooling_kw == 0.0` during active IT operation (IT workload > 2,500 kW). A large data center cannot run at zero cooling without thermal shutdown. These 3 points reflect telemetry logger dropouts and are replaced with `NaN` for linear interpolation.

3. **PUE Ratio Outliers**:
   - By definition, $\text{PUE} = \frac{P_{\text{Total}}}{P_{\text{IT}}}$. In typical high-efficiency facilities, PUE ranges between 1.02 and 1.50. Transient IT power restarts momentarily drop the denominator near zero, driving PUE up to 528.3. We cap PUE at 3.0 to remove extreme scale distortions without altering operational trends.

In [ ]:
pue_keep = ['datetime', 'cooling_kw', 'it_power_kw', 'hvac_kw', 'pump_kw', 'plug_and_light_kw', 'pue', 'ere']
weather_keep = ['datetime', 'outdoor_air_temp', 'outdoor_air_humidity']

df_pue = df_pue[pue_keep]
df_weather = df_weather[weather_keep]

# Treat cooling_kw zeros
zero_cooling = (df_pue['cooling_kw'] <= 0).sum()
print(f"Cooling demand non-positive readings treated: {zero_cooling}")
df_pue.loc[df_pue['cooling_kw'] <= 0, 'cooling_kw'] = np.nan

# Cap extreme PUE spikes
pue_spikes = (df_pue['pue'] > 3.0).sum()
print(f"Extreme PUE spikes (> 3.0) capped: {pue_spikes}")
df_pue.loc[df_pue['pue'] > 3.0, 'pue'] = 3.0

# Linear interpolation for high-frequency sensor gaps (limit = 15 minutes)
df_pue['cooling_kw'] = df_pue['cooling_kw'].interpolate(method='linear', limit=15)
df_weather['outdoor_air_temp'] = df_weather['outdoor_air_temp'].interpolate(method='linear', limit=15)
df_weather['outdoor_air_humidity'] = df_weather['outdoor_air_humidity'].interpolate(method='linear', limit=15)

## 4. Minute-Aligned Synchronous Joining

The PUE and Weather feeds record measurements at slightly offset seconds (e.g., `:07` vs `:45`). Flooding raw timestamps directly into a join would fail to match observations.

We floor all timestamps to the nearest minute (`dt_min = datetime.dt.floor('min')`) and perform an inner join, ensuring synchronous 1-minute alignment without many-to-many cardinality.

In [ ]:
df_pue['dt_min'] = df_pue['datetime'].dt.floor('min')
df_weather['dt_min'] = df_weather['datetime'].dt.floor('min')

df_pue_min = df_pue.drop_duplicates(subset=['dt_min'], keep='first').set_index('dt_min')
df_weather_min = df_weather.drop_duplicates(subset=['dt_min'], keep='first').set_index('dt_min')

merged_1min = pd.merge(
    df_pue_min.drop(columns=['datetime']),
    df_weather_min.drop(columns=['datetime']),
    left_index=True,
    right_index=True,
    how='inner'
)

print(f"Synchronous 1-Minute Aligned Records: {len(merged_1min):,}")
display(merged_1min.head())

proc_dir = os.path.abspath('../data/processed')
os.makedirs(proc_dir, exist_ok=True)
merged_1min.to_csv(os.path.join(proc_dir, 'merged_data.csv'))

## 5. Resampling to Continuous Hourly Cadence (1H)

### Theoretical & Empirical Justification for Hourly Cadence:
1. **Signal-to-Noise Ratio**: Telemetry sampled at 1-minute intervals contains high-frequency transient noise (chiller valve cycling, sensor fluctuations). Taking the hourly mean acts as a low-pass filter, retaining true thermodynamic diurnal cycles.
2. **Computational Tractability**: Fitting SARIMA(p,d,q)(P,D,Q,s) models with seasonal periods of $s=1440$ (daily on 1-min data) is computationally intractable and numerically unstable. With hourly data, daily seasonality corresponds to $s=24$, which is the gold standard in power system forecasting.
3. **Operational Relevance**: Data center cooling equipment dispatch and day-ahead electricity contracts operate on 1-hour schedules.

We construct an explicit, continuous 8,760-hour index spanning **May 1, 2024 00:00:00 to April 30, 2025 23:00:00** ($365 \times 24 = 8,760\text{ hours}$).

In [ ]:
hourly_index = pd.date_range(
    start='2024-05-01 00:00:00',
    end='2025-04-30 23:00:00',
    freq='h',
    name='timestamp'
)

# Resample by taking hourly arithmetic mean
hourly_df = merged_1min.resample('h').mean()

# Reindex to enforce absolute continuous time grid
hourly_df = hourly_df.reindex(hourly_index)

missing_initial = hourly_df['cooling_kw'].isna().sum()
print(f"Missing hours before interpolation: {missing_initial} ({missing_initial/len(hourly_df)*100:.2f}%)")

# Time-weighted interpolation for brief sensor outages
hourly_df = hourly_df.interpolate(method='time', limit=48).bfill().ffill()

assert hourly_df.isna().sum().sum() == 0, "Missing values remain in hourly dataset!"
print(f"Final Clean Continuous Hourly Observations: {len(hourly_df):,} hours")
display(hourly_df.head())

hourly_path = os.path.join(proc_dir, 'hourly_data.csv')
hourly_df.to_csv(hourly_path)
print(f"Successfully saved clean hourly dataset to: {hourly_path}")

## 6. Preprocessing Transition Summary

In [ ]:
summary_table = pd.DataFrame({
    'Processing Stage': [
        'Raw PUE Telemetry',
        'Raw Weather Telemetry',
        'Deduplicated PUE',
        'Deduplicated Weather',
        'Synchronous 1-Minute Merged',
        'Continuous Hourly Series (1H)'
    ],
    'Record Count': [
        f"{len(df_pue_raw):,}",
        f"{len(df_weather_raw):,}",
        f"{len(df_pue):,}",
        f"{len(df_weather):,}",
        f"{len(merged_1min):,}",
        f"{len(hourly_df):,}"
    ],
    'Sampling Frequency': [
        '1 minute (~60 s)',
        '1 minute (~60 s)',
        '1 minute',
        '1 minute',
        '1 minute',
        '1 hour (3,600 s)'
    ],
    'Missingness': [
        '0.0% in target',
        '0.13% in temp',
        '0.0%',
        '0.0%',
        '0.0%',
        '0.0% (100% Complete)'
    ]
})

display(summary_table)
met_dir = os.path.abspath('../results/metrics')
summary_table.to_csv(os.path.join(met_dir, 'phase2_preprocessing_summary.csv'), index=False)

## 7. Diagnostic Visualizations

We generate two validation figures:
1. **High-Frequency vs. Hourly Aggregation Comparison**: Demonstrates how hourly averaging suppresses noise while accurately tracking physical demand peaks.
2. **Complete 1-Year Hourly Overview**: Validates the continuous 8,760-hour trajectories of cooling demand, IT workload, and outdoor temperature across all four seasons.

In [ ]:
# 1-Week Zoom: Raw 1-Min vs. Hourly Mean
zoom_start = '2024-07-08 00:00:00'
zoom_end = '2024-07-15 00:00:00'

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Cooling Demand
axes[0].plot(merged_1min.loc[zoom_start:zoom_end].index,
             merged_1min.loc[zoom_start:zoom_end, 'cooling_kw'],
             color='#a6cee3', lw=1, alpha=0.7, label='Raw 1-Minute Telemetry')
axes[0].plot(hourly_df.loc[zoom_start:zoom_end].index,
             hourly_df.loc[zoom_start:zoom_end, 'cooling_kw'],
             color='#1f78b4', lw=2.5, marker='o', markersize=3, label='Hourly Mean Aggregation (1H)')
axes[0].set_ylabel('Cooling Power Demand (kW)', fontsize=11, fontweight='bold')
axes[0].set_title('High-Frequency Telemetry vs. Hourly Aggregation: Cooling Power Demand (1-Week Zoom)', fontsize=12, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(loc='upper right', frameon=True)

# IT Workload
axes[1].plot(merged_1min.loc[zoom_start:zoom_end].index,
             merged_1min.loc[zoom_start:zoom_end, 'it_power_kw'],
             color='#b2df8a', lw=1, alpha=0.7, label='Raw 1-Minute Workload')
axes[1].plot(hourly_df.loc[zoom_start:zoom_end].index,
             hourly_df.loc[zoom_start:zoom_end, 'it_power_kw'],
             color='#33a02c', lw=2.5, marker='s', markersize=3, label='Hourly Mean Workload (1H)')
axes[1].set_ylabel('IT Equipment Workload (kW)', fontsize=11, fontweight='bold')
axes[1].set_title('High-Frequency Telemetry vs. Hourly Aggregation: IT Workload (1-Week Zoom)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Timestamp (Mountain Time)', fontsize=11, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='upper right', frameon=True)

plt.tight_layout()
fig_dir = os.path.abspath('../results/figures')
fig1_path = os.path.join(fig_dir, 'phase2_hourly_aggregation_comparison.png')
plt.savefig(fig1_path, dpi=300)
plt.show()
print(f"Saved: {fig1_path}")

In [ ]:
# Complete 1-Year Clean Hourly Series Overview
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

axes[0].plot(hourly_df.index, hourly_df['cooling_kw'], color='#1f78b4', lw=1, alpha=0.85)
axes[0].set_ylabel('Cooling Power (kW)', fontsize=11, fontweight='bold')
axes[0].set_title('Clean Continuous Hourly Time Series: Cooling Power Demand (cooling_kw)', fontsize=12, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].plot(hourly_df.index, hourly_df['it_power_kw'], color='#33a02c', lw=1, alpha=0.85)
axes[1].set_ylabel('IT Workload (kW)', fontsize=11, fontweight='bold')
axes[1].set_title('Clean Continuous Hourly Time Series: IT Equipment Workload (it_power_kw)', fontsize=12, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.6)

axes[2].plot(hourly_df.index, hourly_df['outdoor_air_temp'], color='#ff7f00', lw=1, alpha=0.85)
axes[2].set_ylabel('Outdoor Temp (°F)', fontsize=11, fontweight='bold')
axes[2].set_title('Clean Continuous Hourly Time Series: Outdoor Dry-Bulb Temperature', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Timestamp (Hourly, May 2024 – April 2025)', fontsize=11, fontweight='bold')
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
fig2_path = os.path.join(fig_dir, 'phase2_hourly_clean_overview.png')
plt.savefig(fig2_path, dpi=300)
plt.show()
print(f"Saved: {fig2_path}")